# Step 8: RAG Pipeline for Evidence-Based Risk Explanations

This notebook implements a Retrieval-Augmented Generation (RAG) pipeline that provides
human-readable, evidence-based explanations for GNN-predicted health risks.

**Components:**
1. **FAISS Vector Index**: Semantic search over PubMed evidence
2. **Neo4j Knowledge Graph**: Structured ingredient-disease queries  
3. **BioMistral-7B (Ollama)**: GPU-accelerated explanation generation

**Prerequisites:**
- Neo4j running with knowledge graph from Step 5
- Ollama with `jsk/bio-mistral` model in Docker
- GNN predictions from Step 7 (`models/ingredient_risk_predictions.csv`)

In [1]:
# Imports and setup
import os
import sys
import json
import requests
import numpy as np
import pandas as pd
from pathlib import Path
from collections import defaultdict
from typing import List, Dict, Any

# Add src to path
sys.path.insert(0, '../src')

# Check GPU
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

PyTorch version: 2.10.0+cpu
CUDA available: False


## 1. Configuration

In [2]:
# Configuration - matching Step 7 paths
CONFIG = {
    # Data paths (same as Step 7)
    'kg_path': '../data/pre-processed/kg/',
    'model_path': '../models/',
    
    # Neo4j settings (same as Step 5)
    'neo4j_uri': 'neo4j://127.0.0.1:7687',
    'neo4j_user': 'neo4j',
    'neo4j_password': 'password',
    
    # Ollama settings (Docker with GPU)
    'ollama_url': 'http://localhost:11434',
    'ollama_model': 'jsk/bio-mistral',  # BioMistral-7B
    
    # FAISS settings
    'faiss_index_path': '../models/faiss_pubmed.index',
    'faiss_metadata_path': '../models/faiss_pubmed_metadata.json',
    
    # Target diseases (same as Step 7)
    'target_diseases': [
        'diabetes',
        'obesity', 
        'hypertension',
        'cancer',
        'cardiovascular disease'
    ],
    
    # Search parameters
    'top_k_faiss': 10,
    'top_k_neo4j': 20,
}

# Create model directory
os.makedirs(CONFIG['model_path'], exist_ok=True)
print("Configuration loaded!")
print(f"  PubMed data: {CONFIG['kg_path']}")
print(f"  Neo4j: {CONFIG['neo4j_uri']}")
print(f"  Ollama: {CONFIG['ollama_url']} (model: {CONFIG['ollama_model']})")

Configuration loaded!
  PubMed data: ../data/pre-processed/kg/
  Neo4j: neo4j://127.0.0.1:7687
  Ollama: http://localhost:11434 (model: jsk/bio-mistral)


## 2. Load Data from Previous Steps

In [3]:
# Load PubMed triples (from Step 5)
print("Loading PubMed knowledge graph triples...")
pubmed_file = os.path.join(CONFIG['kg_path'], 'pubmed_kg_triples.csv')
pubmed_df = pd.read_csv(pubmed_file)
print(f"Loaded {len(pubmed_df):,} PubMed triples")
print(f"Columns: {pubmed_df.columns.tolist()}")
pubmed_df.head()

Loading PubMed knowledge graph triples...
Loaded 30,567 PubMed triples
Columns: ['subject', 'predicate', 'object', 'pmid', 'title', 'year', 'journal']


,subject,predicate,object,pmid,title,year,journal
0,protein,RELATES_TO,"diabetes, insulin resistance, cardiovascular d...",41390803,Association between triglyceride-glucose index...,2025,Cardiovascular diabetology
1,protein,RELATES_TO,alzheimer,41390778,Multi-task learning identifies shared genetic ...,2025,Scientific reports
2,protein,RELATES_TO,"diabetes, obesity, diabetes mellitus",41390701,Obesity concurrent with gestational diabetes m...,2025,Scientific reports
3,protein,RELATES_TO,"diabetes, type 2 diabetes",41390575,The RNA-binding protein CPEB1 marks healthy ad...,2025,Scientific reports
4,protein,RELATES_TO,"inflammation, cardiovascular disease, stroke, ...",41390442,Genetic insights into 5-LOX-activating protein...,2025,Human genomics


In [4]:
# Load GNN predictions from Step 7
print("Loading GNN predictions from Step 7...")
gnn_pred_file = os.path.join(CONFIG['model_path'], 'ingredient_risk_predictions.csv')

if os.path.exists(gnn_pred_file):
    gnn_predictions_df = pd.read_csv(gnn_pred_file)
    print(f"Loaded {len(gnn_predictions_df):,} ingredient predictions")
    print(f"Columns: {gnn_predictions_df.columns.tolist()}")
    display(gnn_predictions_df.head())
else:
    print(f"⚠️ GNN predictions not found at {gnn_pred_file}")
    print("   Run Step 7 first to generate predictions.")
    gnn_predictions_df = None

Loading GNN predictions from Step 7...
Loaded 225,440 ingredient predictions
Columns: ['ingredient', 'diabetes_risk', 'obesity_risk', 'hypertension_risk', 'cancer_risk', 'cardiovascular disease_risk']


,ingredient,diabetes_risk,obesity_risk,hypertension_risk,cancer_risk,cardiovascular disease_risk
0,TANGY CHIPOTLE SAUCE,0.700576,0.680452,0.139485,0.410091,0.634222
1,RED MARASCHINO CHERRIES,0.454736,0.436938,0.579564,0.572626,0.728553
2,DEHYDRATED RED BELL PEPPER),0.790574,0.682114,0.626193,0.468933,0.757938
3,BACON BASE,0.878756,0.753918,0.838016,0.358697,0.899820
4,BBQ COATED PEANUTS,0.722019,0.736715,0.730791,0.352214,0.778813


## 3. Initialize RAG Pipeline Components

In [5]:
# Import RAG pipeline components
from rag_pipeline import (
    RAGConfig, RAGPipeline, 
    FAISSIndexBuilder, Neo4jConnector, OllamaClient,
    EvidenceAggregator, ExplanationGenerator
)

# Create configuration
rag_config = RAGConfig(
    neo4j_uri=CONFIG['neo4j_uri'],
    neo4j_user=CONFIG['neo4j_user'],
    neo4j_password=CONFIG['neo4j_password'],
    ollama_url=CONFIG['ollama_url'],
    ollama_model=CONFIG['ollama_model'],
    faiss_index_path=CONFIG['faiss_index_path'],
    faiss_metadata_path=CONFIG['faiss_metadata_path'],
    top_k_faiss=CONFIG['top_k_faiss'],
    top_k_neo4j=CONFIG['top_k_neo4j'],
    target_diseases=CONFIG['target_diseases']
)

print("RAG Configuration created!")

c:\Work\hassan work\food work\Research Project_2\Research Project\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Work\hassan work\food work\Research Project_2\Research Project\venv\lib\site-packages\torchvision\io\image.py:13: UserWarning: Failed to load image Python extension: 'Could not load this library: C:\Work\hassan work\food work\Research Project_2\Research Project\venv\Lib\site-packages\torchvision\image.pyd'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(


RAG Configuration created!


## 4. Build FAISS Vector Index

In [7]:
# Build or load FAISS index from PubMed data
faiss_builder = FAISSIndexBuilder(rag_config)

if os.path.exists(CONFIG['faiss_index_path']):
    print("Loading existing FAISS index...")
    faiss_builder.load()
else:
    print("Building FAISS index from PubMed data...")
    faiss_builder.build_index(pubmed_file)
    
print(f"\n✅ FAISS index ready with {len(faiss_builder.metadata):,} entries")

Building FAISS index from PubMed data...
Building FAISS index from: ../data/pre-processed/kg/pubmed_kg_triples.csv
  Loaded 30,567 PubMed records
  Generating embeddings...
Loading embedding model: all-MiniLM-L6-v2


`SentenceTransformer._target_device` has been deprecated, please use `SentenceTransformer.device` instead.
`SentenceTransformer._target_device` has been deprecated, please use `SentenceTransformer.device` instead.


  Using device: cpu


Batches: 100%|██████████| 956/956 [02:17<00:00,  6.97it/s]


  Building FAISS index...
  Using CPU FAISS index
  Index built with 30,567 vectors
  Saved index to: ../models/faiss_pubmed.index
  Saved metadata to: ../models/faiss_pubmed_metadata.json

✅ FAISS index ready with 30,567 entries


In [8]:
# Test FAISS semantic search
print("Testing FAISS semantic search...\n")

test_queries = ["sugar diabetes", "saturated fat heart disease", "protein obesity"]

for query in test_queries:
    print(f"🔍 Query: '{query}'")
    results = faiss_builder.search(query, k=3)
    for i, r in enumerate(results, 1):
        print(f"   {i}. [{r['similarity_score']:.3f}] {r['subject']} → {r['object'][:50]}...")
        print(f"      PMID:{r['pmid']} | {r['journal'][:40]}...")
    print()

Testing FAISS semantic search...

🔍 Query: 'sugar diabetes'
   1. [0.731] added sugars → diabetes, type 2 diabetes, obesity...
      PMID:38177563 | Diabetologia...
   2. [0.706] added sugars → diabetes, insulin resistance, cardiovascular disea...
      PMID:38648172 | FP essentials...
   3. [0.696] maltose → diabetes, diabetes mellitus...
      PMID:39714134 | Journal of materials chemistry. B...

🔍 Query: 'saturated fat heart disease'
   1. [0.701] saturated fatty acids → cancer, heart disease...
      PMID:39875911 | Lipids in health and disease...
   2. [0.663] trans fatty acids → heart disease...
      PMID:38303014 | Journal of health, population, and nutri...
   3. [0.650] saturated fatty acids → cardiovascular disease, inflammation, chronic infl...
      PMID:39796562 | Nutrients...

🔍 Query: 'protein obesity'
   1. [0.696] casein → insulin resistance, obesity...
      PMID:41156477 | Nutrients...
   2. [0.682] egg → insulin resistance, obesity...
      PMID:41156477 | Nutrient

## 5. Connect to Neo4j Knowledge Graph

In [9]:
# Connect to Neo4j
neo4j_conn = Neo4jConnector(rag_config)

✅ Connected to Neo4j at neo4j://127.0.0.1:7687


In [10]:
# Test Neo4j queries
print("Testing Neo4j knowledge graph queries...\n")

# Get disease statistics
stats = neo4j_conn.get_disease_statistics()
print("Top 10 diseases by linked ingredients:")
for i, (disease, count) in enumerate(list(stats.items())[:10], 1):
    print(f"  {i:2}. {disease[:50]:50s} - {count:,} ingredients")

Testing Neo4j knowledge graph queries...

Top 10 diseases by linked ingredients:
   1. cancer                                             - 212 ingredients
   2. inflammation                                       - 210 ingredients
   3. diabetes                                           - 189 ingredients
   4. obesity                                            - 187 ingredients
   5. cancer, breast cancer                              - 181 ingredients
   6. depression                                         - 169 ingredients
   7. colorectal cancer, cancer                          - 158 ingredients
   8. cardiovascular disease                             - 150 ingredients
   9. diabetes, diabetes mellitus                        - 149 ingredients
  10. diabetes, type 2 diabetes                          - 149 ingredients


In [11]:
# Query ingredient-disease relationships
test_ingredients = ["protein", "sugar", "saturated fatty acids"]

for ingredient in test_ingredients:
    print(f"\n📊 Ingredient: '{ingredient}'")
    results = neo4j_conn.get_ingredient_diseases_fuzzy(ingredient, limit=5)
    
    if results:
        for r in results[:3]:
            print(f"   → {r['disease'][:50]}...")
            print(f"     PMID:{r['pmid']} | {r['title'][:50]}...")
    else:
        print("   No direct relationships found")


📊 Ingredient: 'protein'
   → diabetes, insulin resistance, cardiovascular disea...
     PMID:41390803 | Association between triglyceride-glucose index com...
   → alzheimer...
     PMID:40325994 | Immunocal...
   → alzheimer...
     PMID:41390778 | Multi-task learning identifies shared genetic risk...

📊 Ingredient: 'sugar'
   → diabetes, obesity, diabetes mellitus...
     PMID:38367172 | Mapping Lifestyle Interventions for Gestational Di...
   → diabetes, type 2 diabetes...
     PMID:40895866 | Exercise and Lifestyle Modification in the Managem...
   → diabetes, diabetes mellitus...
     PMID:41223884 | Lifestyle prescriptions for diabetes management in...

📊 Ingredient: 'saturated fatty acids'
   → alzheimer...
     PMID:40295359 | Ceramides may Play a Central Role in the Pathogene...
   → alzheimer...
     PMID:41326444 | Introduction of Cistanche phelypaea fatty acids as...
   → diabetes, type 2 diabetes...
     PMID:41007350 | Tissue-Specific Differences in Fatty Acid Content ...

## 6. Test Ollama/BioMistral Connection

In [12]:
# Test Ollama connection
ollama_client = OllamaClient(rag_config)

# Quick test generation
print("Testing BioMistral-7B generation...")
test_response = ollama_client.generate(
    "Briefly explain how sugar consumption affects diabetes risk.",
    max_tokens=100
)
print(f"\nResponse: {test_response[:300]}...")

✅ Ollama connected, model 'jsk/bio-mistral' available
Testing BioMistral-7B generation...

Response:  Sugar is one of the main sources of energy and sweetness in our diet, and it can come from beverages such as soda or fruit juices, from sugary snacks like candy or cookies, or from starchy foods that are quickly digested to glucose, such as white rice. The World Health Organization recommends an in...


## 7. Initialize Full RAG Pipeline

In [13]:
# Initialize complete RAG pipeline
pipeline = RAGPipeline(rag_config)

# Load or build FAISS index
if os.path.exists(CONFIG['faiss_index_path']):
    pipeline.load_index()
else:
    pipeline.build_index(pubmed_file)

Initializing RAG Pipeline...
✅ Connected to Neo4j at neo4j://127.0.0.1:7687
✅ Ollama connected, model 'jsk/bio-mistral' available
RAG Pipeline initialized.

Loading FAISS index from: ../models/faiss_pubmed.index
  Loaded index to CPU
  Loaded 30,567 metadata entries


## 8. Generate Risk Explanations with GNN Predictions

In [14]:
def get_gnn_predictions_for_ingredient(ingredient_name: str) -> Dict[str, float]:
    """
    Get GNN predictions for an ingredient from Step 7 output.
    """
    if gnn_predictions_df is None:
        return {}
    
    # Find ingredient (case-insensitive partial match)
    mask = gnn_predictions_df['ingredient'].str.lower().str.contains(ingredient_name.lower())
    matches = gnn_predictions_df[mask]
    
    if len(matches) == 0:
        return {}
    
    # Get first match
    row = matches.iloc[0]
    
    # Extract disease risk columns
    predictions = {}
    for disease in CONFIG['target_diseases']:
        col_name = f"{disease}_risk"
        if col_name in row:
            predictions[disease] = float(row[col_name])
    
    return predictions

# Test
test_preds = get_gnn_predictions_for_ingredient("protein")
print("GNN predictions for 'protein':")
for disease, prob in test_preds.items():
    risk_level = "HIGH" if prob > 0.7 else "MODERATE" if prob > 0.4 else "LOW"
    print(f"  {disease}: {prob:.3f} ({risk_level})")

GNN predictions for 'protein':
  diabetes: 0.831 (HIGH)
  obesity: 0.753 (HIGH)
  hypertension: 0.308 (LOW)
  cancer: 0.424 (MODERATE)
  cardiovascular disease: 0.570 (MODERATE)


In [15]:
# Analyze an ingredient with full RAG pipeline + GNN predictions
ingredient_to_analyze = "protein"

print(f"🔬 Analyzing: {ingredient_to_analyze}")
print("=" * 60)

# Get GNN predictions
gnn_preds = get_gnn_predictions_for_ingredient(ingredient_to_analyze)

# Run RAG analysis
result = pipeline.analyze_ingredient(ingredient_to_analyze, gnn_predictions=gnn_preds)

# Display results
print("\n📊 GNN Risk Predictions:")
for disease, prob in gnn_preds.items():
    risk_level = "HIGH" if prob > 0.7 else "MODERATE" if prob > 0.4 else "LOW"
    print(f"  {disease.upper()}: {prob:.1%} ({risk_level})")

print("\n📈 Evidence Summary:")
for disease, summary in result['evidence_summary'].items():
    if summary['total_evidence'] > 0:
        print(f"  {disease.upper()}: {summary['total_evidence']} studies")

print("\n📝 Generated Explanation:")
print("-" * 60)
print(result['explanation'])

🔬 Analyzing: protein
Analyzing ingredient: protein
  Searching FAISS index...
Loading embedding model: all-MiniLM-L6-v2


`SentenceTransformer._target_device` has been deprecated, please use `SentenceTransformer.device` instead.
`SentenceTransformer._target_device` has been deprecated, please use `SentenceTransformer.device` instead.


  Using device: cpu
    Found 10 similar evidence
  Querying Neo4j knowledge graph...
    Found 20 graph relationships
  Aggregating evidence...
  Generating explanation with BioMistral-7B...

📊 GNN Risk Predictions:
  DIABETES: 83.1% (HIGH)
  OBESITY: 75.3% (HIGH)
  HYPERTENSION: 30.8% (LOW)
  CANCER: 42.4% (MODERATE)
  CARDIOVASCULAR DISEASE: 57.0% (MODERATE)

📈 Evidence Summary:
  DIABETES: 10 studies
  HYPERTENSION: 2 studies
  CANCER: 8 studies
  OBESITY: 2 studies
  CARDIOVASCULAR DISEASE: 4 studies

📝 Generated Explanation:
------------------------------------------------------------
 Protein is an essential nutrient that plays a crucial role in maintaining health and wellbeing. However, excessive protein consumption has been linked to various adverse health outcomes such as diabetes, hypertension, cancer, obesity, and cardiovascular disease. The scientific evidence suggests that high-protein diets can lead to insulin resistance and glucose intolerance, which are risk factors fo

## 9. Batch Analysis of High-Risk Ingredients

In [16]:
# Get top 10 high-risk ingredients from GNN predictions
if gnn_predictions_df is not None:
    # Calculate average risk across all diseases
    risk_cols = [col for col in gnn_predictions_df.columns if col.endswith('_risk')]
    gnn_predictions_df['avg_risk'] = gnn_predictions_df[risk_cols].mean(axis=1)
    
    # Get top 10
    top_ingredients = gnn_predictions_df.nlargest(10, 'avg_risk')
    
    print("Top 10 High-Risk Ingredients (from GNN):")
    print("=" * 60)
    for i, (_, row) in enumerate(top_ingredients.iterrows(), 1):
        print(f"{i:2}. {row['ingredient'][:40]:40s} (avg risk: {row['avg_risk']:.3f})")
else:
    print("GNN predictions not available. Run Step 7 first.")

Top 10 High-Risk Ingredients (from GNN):
 1. BARLEY GRASS POWDER†                     (avg risk: 0.881)
 2. AND XANTHUM GUM                          (avg risk: 0.880)
 3. wheat                                    (avg risk: 0.880)
 4. SODIUM NITRITE. SWEET APPLE CHIPOTLE BBQ (avg risk: 0.880)
 5. US CERT. COLORS INCLUDING: FD&C BLUE NO. (avg risk: 0.879)
 6. NON-GMO WHITE                            (avg risk: 0.878)
 7. GREEK OLIVES                             (avg risk: 0.878)
 8. folate                                   (avg risk: 0.877)
 9. probiotic                                (avg risk: 0.877)
10. SODIUM BICARBONATE ADDED AS RAISING AGEN (avg risk: 0.877)


In [17]:
# Analyze top 3 high-risk ingredients
if gnn_predictions_df is not None:
    ingredients_to_analyze = top_ingredients['ingredient'].head(3).tolist()
    
    all_results = []
    
    for ingredient in ingredients_to_analyze:
        print(f"\n{'=' * 60}")
        print(f"🔬 Analyzing: {ingredient}")
        print("=" * 60)
        
        gnn_preds = get_gnn_predictions_for_ingredient(ingredient)
        result = pipeline.analyze_ingredient(ingredient, gnn_predictions=gnn_preds)
        
        # Show brief summary
        print(f"\nExplanation preview:")
        preview = result['explanation'][:400] + "..." if len(result['explanation']) > 400 else result['explanation']
        print(preview)
        
        all_results.append(result)


🔬 Analyzing: BARLEY GRASS POWDER†
Analyzing ingredient: BARLEY GRASS POWDER†
  Searching FAISS index...
    Found 10 similar evidence
  Querying Neo4j knowledge graph...
    Found 0 graph relationships
  Aggregating evidence...
  Generating explanation with BioMistral-7B...

Explanation preview:
 Based on the available evidence, barley grass powder is associated with a high risk of obesity, cancer, cardiovascular disease, hypertension, and diabetes. The GNN (Global Nutrition Network) Risk Prediction algorithm was used to determine the risk level for each health condition based on the scientific literature. For obesity, the GNN Risk Prediction algorithm determined a 89.2% risk, which is co...

🔬 Analyzing: AND XANTHUM GUM
Analyzing ingredient: AND XANTHUM GUM
  Searching FAISS index...
    Found 10 similar evidence
  Querying Neo4j knowledge graph...
    Found 0 graph relationships
  Aggregating evidence...
  Generating explanation with BioMistral-7B...

Explanation preview:
 Based on 

In [18]:
# Export analysis results
if gnn_predictions_df is not None:
    # Analyze more ingredients for export
    export_ingredients = top_ingredients['ingredient'].head(10).tolist()
    
    export_data = []
    
    for ingredient in export_ingredients:
        print(f"Analyzing: {ingredient}...")
        gnn_preds = get_gnn_predictions_for_ingredient(ingredient)
        result = pipeline.analyze_ingredient(ingredient, gnn_predictions=gnn_preds)
        
        for disease, evidence in result['disease_evidence'].items():
            export_data.append({
                'ingredient': ingredient,
                'disease': disease,
                'gnn_risk': gnn_preds.get(disease, 0),
                'evidence_count': len(evidence),
                'top_pmid': evidence[0]['pmid'] if evidence else None
            })
    
    # Create DataFrame and save
    export_df = pd.DataFrame(export_data)
    export_path = os.path.join(CONFIG['model_path'], 'rag_analysis_results.csv')
    export_df.to_csv(export_path, index=False)
    
    print(f"\n✅ Results saved to: {export_path}")
    display(export_df.head(10))

Analyzing: BARLEY GRASS POWDER†...
Analyzing ingredient: BARLEY GRASS POWDER†
  Searching FAISS index...
    Found 10 similar evidence
  Querying Neo4j knowledge graph...
    Found 0 graph relationships
  Aggregating evidence...
  Generating explanation with BioMistral-7B...
Analyzing: AND XANTHUM GUM...
Analyzing ingredient: AND XANTHUM GUM
  Searching FAISS index...
    Found 10 similar evidence
  Querying Neo4j knowledge graph...
    Found 0 graph relationships
  Aggregating evidence...
  Generating explanation with BioMistral-7B...
Analyzing: wheat...
Analyzing ingredient: wheat
  Searching FAISS index...
    Found 10 similar evidence
  Querying Neo4j knowledge graph...
    Found 20 graph relationships
  Aggregating evidence...
  Generating explanation with BioMistral-7B...
Analyzing: SODIUM NITRITE. SWEET APPLE CHIPOTLE BBQ SAUCE  WATER...
Analyzing ingredient: SODIUM NITRITE. SWEET APPLE CHIPOTLE BBQ SAUCE  WATER
  Searching FAISS index...
    Found 10 similar evidence
  Querying

,ingredient,disease,gnn_risk,evidence_count,top_pmid
0,BARLEY GRASS POWDER†,obesity,0.891578,2,39121780
1,BARLEY GRASS POWDER†,cancer,0.739203,3,38501834
2,BARLEY GRASS POWDER†,cardiovascular disease,0.907258,3,38501834
3,BARLEY GRASS POWDER†,hypertension,0.957830,2,37430980
4,BARLEY GRASS POWDER†,diabetes,0.907306,2,39640645
5,AND XANTHUM GUM,diabetes,0.959188,1,29565725
6,AND XANTHUM GUM,cardiovascular disease,0.954454,2,29565725
7,AND XANTHUM GUM,cancer,0.626725,4,32583796
8,wheat,obesity,0.752983,3,40537902
9,wheat,diabetes,0.831018,4,38571920


## 11. Cleanup

In [19]:
# Close connections
pipeline.close()
neo4j_conn.close()
print("\nRAG Pipeline closed successfully!")


RAG Pipeline closed successfully!


## Summary

This notebook demonstrated the complete RAG pipeline for evidence-based health risk explanations:

1. Built FAISS vector index from PubMed articles
2. Queried Neo4j knowledge graph for ingredient-disease relationships  
3. Generated explanations using BioMistral-7B via Ollama (GPU-accelerated)
4. Integrated with GNN predictions from Step 7

**Output Files:**
- `models/faiss_pubmed.index` - Vector index for semantic search
- `models/faiss_pubmed_metadata.json` - Metadata for indexed documents
- `models/rag_analysis_results.csv` - Analysis results with evidence
